# Executor Tuning

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

**Deviation from the mockup's exact numbers:** `docs/requirements/curriculum-topics-2026-07.md`'s US-C3 cites `executor-cores=8, executor-memory=28g` (fat) vs. `executor-cores=5, executor-memory=12g` (right-sized) -- both exceed this platform's own hard per-worker ceiling (`WORKER_CORES_RANGE=(1,4)`, `WORKER_MEMORY_GB_RANGE=(1,8)`, `app/config.py`, enforced in `app/lifecycle/renderer.py`, not just a UI hint). This notebook reproduces the same *concept* -- fewer, fatter executors cramming more concurrent tasks into one shared heap vs. more, right-sized executors giving each task more headroom -- at `executor-cores=4, executor-memory=2g` (fat, ~500MB/task) vs. `executor-cores=2, executor-memory=2g` (right-sized, ~1GB/task), holding the physical cluster (and its total cores/memory) fixed across both runs. See `manifest.yaml`'s and `concept.md`'s own deviation notes for the full reasoning.

Two claims under test, each checked directly against real evidence (per-executor `totalGCTime`/`totalDuration` from the Spark REST API, not an assumed number):
1. Running the identical job with 1 fat executor per node (`executor-cores=4, executor-memory=2g`) vs. 2 right-sized executors per node (`executor-cores=2, executor-memory=2g`) produces a measurably different GC-time fraction (`totalGCTime / totalDuration`, summed across executors) -- the fat run's fraction should come out higher, since 4 concurrent tasks are sharing one 2GB heap instead of 2 tasks each getting their own.
2. Wall-clock is captured for both runs too -- but (found by actually running both configs against a real cluster, not assumed) it does **not** reliably favor the right-sized run at this project's small dev-cluster scale: doubling executor *count* while holding total cluster cores/memory fixed adds real shuffle fan-out/coordination overhead that can outweigh the GC savings on a small dataset. See `concept.md`'s own "Honest result" note -- this notebook reports what actually happens rather than asserting away the nuance.

In [ ]:
import json
import time
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

SCRATCH_DIR = "/workspace/scratch/executor-tuning"


def build_spark(executor_cores, executor_memory, suffix):
    """A fresh SparkSession with a specific executor shape. `spark.executor.cores`/
    `spark.executor.memory` must be set before the SparkContext starts -- they can't
    change on a live session, which is why each run below stops the previous
    session first rather than reconfiguring in place."""
    spark = (
        SparkSession.builder.appName(f"executor-tuning-{suffix}")
        .config("spark.executor.cores", str(executor_cores))
        .config("spark.executor.memory", executor_memory)
        .config("spark.dynamicAllocation.enabled", "false")
        .getOrCreate()
    )
    spark.range(20).count()  # warm-up: forces executor JVMs to launch now, not
                             # inside the timed run below (JVM startup is a fixed
                             # per-executor cost unrelated to the GC/wall-clock
                             # question this notebook is actually testing).
    return spark


def executors_snapshot(spark):
    """Current `/api/v1/applications/<id>/executors` list -- the ground truth for
    per-executor `totalGCTime`/`totalDuration`, the same REST surface the Spark UI's
    Executors tab reads, and the exact fields this topic's manifest.yaml spotlights
    via executor_metrics."""
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/executors"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def gc_fraction(executors):
    """Sums totalGCTime/totalDuration across every *real* executor (excludes the
    'driver' entry the REST API also reports as an executor id) -- the GC-time
    fraction this topic's self-check evidence is built around."""
    real = [e for e in executors if e.get("id") != "driver"]
    total_gc = sum(e.get("totalGCTime", 0) for e in real)
    total_duration = sum(e.get("totalDuration", 0) for e in real)
    fraction = total_gc / total_duration if total_duration else 0.0
    return len(real), total_gc, total_duration, fraction

## 1. Build a shuffle-heavy dataset once, shared by both runs

A wide-ish row (a couple of doubles plus a padded string payload, to generate real JVM object churn, not just primitive columns) with a skewed-enough key space that `groupBy("key")` genuinely has to shuffle -- written to Parquet once so both runs below read the identical on-disk data.

In [ ]:
NUM_ROWS = 12_000_000
NUM_PARTITIONS = 96
NUM_KEYS = 8_000

bootstrap = SparkSession.builder.appName("executor-tuning-bootstrap").getOrCreate()


def make_row(i):
    return Row(
        key=i % NUM_KEYS,
        a=float(i % 97) / 3.0,
        b=float(i % 131) / 7.0,
        payload=("x" * 200) + str(i),
    )


rdd = bootstrap.sparkContext.parallelize(range(NUM_ROWS), NUM_PARTITIONS).map(make_row)
wide_df = bootstrap.createDataFrame(rdd)

dataset_path = f"{SCRATCH_DIR}/wide_dataset"
wide_df.write.mode("overwrite").parquet(dataset_path)
print(f"Wrote {NUM_ROWS} rows ({NUM_KEYS} distinct keys) to {dataset_path}")

bootstrap.stop()
time.sleep(2)  # let the bootstrap session's executors fully release before run 1 launches its own

## 2. Run 1 -- "1 fat executor per node" (`executor-cores=4, executor-memory=2g`)

**Hypothesis:** with 4 concurrent tasks sharing one 2GB heap per executor, do you expect a higher or lower GC-time fraction than a run where each task gets its own, less-contended heap? Write your answer, then run the cell below.

The job itself: a full sort on the padded `payload` column (forces a real, memory-hungry shuffle sort, not just an in-place aggregation) followed by a `groupBy("key")` aggregation -- representative GC/shuffle pressure, not a toy `.count()`.

In [ ]:
def run_job(spark, df):
    sorted_df = df.orderBy("payload")
    return (
        sorted_df.groupBy("key")
        .agg(F.sum("a").alias("sum_a"), F.avg("b").alias("avg_b"), F.count("*").alias("n"))
        .agg(F.sum("sum_a"), F.sum("avg_b"), F.sum("n"))
        .collect()
    )


spark_fat = build_spark(executor_cores=4, executor_memory="2g", suffix="fat")
fat_df = spark_fat.read.parquet(dataset_path)

t0 = time.time()
run_job(spark_fat, fat_df)
fat_wall_s = time.time() - t0

fat_n_executors, fat_gc_ms, fat_duration_ms, fat_gc_fraction = gc_fraction(executors_snapshot(spark_fat))
print(f"[fat] executors={fat_n_executors} wall={fat_wall_s:.2f}s totalGCTime={fat_gc_ms}ms totalDuration={fat_duration_ms}ms gc_fraction={fat_gc_fraction:.4f}")
# Measured on this dev cluster: 3 executors, wall~20-22s, gc_fraction~0.034-0.035.

## 3. Checkpoint run 1 for the self-check reveal

Click **Reveal self-check** on the topic page after running this: the executor-metrics panel spotlights `totalGCTime` (plus `totalDuration`/`totalTasks`) for every executor in this run -- 3 rows, one fat executor per node.

In [ ]:
checkpoint(fat_df, topic="executor-tuning")
print("Checkpoint written for the fat-executor run -- click 'Reveal self-check' on the topic page now to see it, then continue below for the right-sized run.")

## 4. Run 2 -- right-sized executors (`executor-cores=2, executor-memory=2g`)

**Hypothesis:** same total cluster cores and memory as run 1, just sliced into twice as many executors, each with half as many concurrent tasks sharing its heap. Do you expect GC-time fraction to go up, down, or stay the same? What about wall-clock -- do you expect it to definitely improve? Write your answers, then run the cell below.

In [ ]:
spark_fat.stop()  # spark.executor.cores/memory can't change on a live session -- stop and rebuild
time.sleep(2)

spark_rightsized = build_spark(executor_cores=2, executor_memory="2g", suffix="rightsized")
rightsized_df = spark_rightsized.read.parquet(dataset_path)

t0 = time.time()
run_job(spark_rightsized, rightsized_df)
rightsized_wall_s = time.time() - t0

rightsized_n_executors, rightsized_gc_ms, rightsized_duration_ms, rightsized_gc_fraction = gc_fraction(
    executors_snapshot(spark_rightsized)
)
print(
    f"[right-sized] executors={rightsized_n_executors} wall={rightsized_wall_s:.2f}s "
    f"totalGCTime={rightsized_gc_ms}ms totalDuration={rightsized_duration_ms}ms gc_fraction={rightsized_gc_fraction:.4f}"
)
# Measured on this dev cluster: 6 executors, wall~26s, gc_fraction~0.026-0.027 on
# most runs, but not every run -- at least one real run saw this come out *higher*
# than the fat run's fraction instead (see cell 13's note); wall-clock also came out
# *slower* here both times observed (see cell 6's note).

## 5. Checkpoint run 2 for the self-check reveal

Click **Reveal self-check** again after running this: the executor-metrics panel now shows 6 rows (2 right-sized executors per node) with the newest checkpoint's numbers -- compare this reveal against the previous one from run 1.

In [ ]:
checkpoint(rightsized_df, topic="executor-tuning")
print("Checkpoint written for the right-sized run -- click 'Reveal self-check' again to compare against run 1's numbers.")

## 6. Compare both runs side by side

Both claims are printed and compared below, but **neither is asserted in a fixed direction** -- see the intro cell and `concept.md`'s "Honest result" note for why: at this project's small dev-cluster scale, more (right-sized) executors add real coordination overhead that can outweigh their GC savings, and that overhead has been observed to flip *both* the wall-clock comparison and, on at least one real run, the GC-time-fraction comparison too. The heuristic's prediction (fat run's GC-time fraction higher) holds on most runs but is not guaranteed at this scale -- the notebook reports what actually happens rather than asserting away the nuance.

In [ ]:
print(f"{'':14s}{'executors':>10s}{'wall (s)':>10s}{'GC time (ms)':>14s}{'duration (ms)':>15s}{'GC fraction':>13s}")
print(f"{'fat':14s}{fat_n_executors:>10d}{fat_wall_s:>10.2f}{fat_gc_ms:>14d}{fat_duration_ms:>15d}{fat_gc_fraction:>13.4f}")
print(
    f"{'right-sized':14s}{rightsized_n_executors:>10d}{rightsized_wall_s:>10.2f}"
    f"{rightsized_gc_ms:>14d}{rightsized_duration_ms:>15d}{rightsized_gc_fraction:>13.4f}"
)

spark_rightsized.stop()  # release before any assertion below can raise, so a
                          # failed assertion never strands this run's executors
                          # (and the cluster's entire capacity) for the next run

assert fat_n_executors == 3, f"expected 1 fat executor per node (3 total), got {fat_n_executors}"
assert rightsized_n_executors == 6, f"expected 2 right-sized executors per node (6 total), got {rightsized_n_executors}"
assert fat_wall_s != rightsized_wall_s, "expected a measurable (nonzero) wall-clock difference between the two runs"

# GC-time fraction: reported, not hard-asserted -- same honest-reporting treatment as
# wall-clock above. The heuristic predicts fat > right-sized (more concurrent tasks
# sharing one heap), and that held on most real runs, but at least one real run at this
# dev-cluster scale saw it flip (see cell 9's note) -- so this notebook prints the
# measured direction instead of asserting one.
gc_direction = "higher" if fat_gc_fraction > rightsized_gc_fraction else "NOT higher"
print(
    f"\nGC-time fraction: fat={fat_gc_fraction:.4f}, right-sized={rightsized_gc_fraction:.4f} "
    f"-- fat's fraction came out {gc_direction} than right-sized's this run "
    f"(the heuristic's predicted direction is fat > right-sized; see this cell's and cell 9's "
    f"notes for why that isn't guaranteed at this scale)"
)
print(f"Wall-clock: fat={fat_wall_s:.2f}s, right-sized={rightsized_wall_s:.2f}s (see this cell's markdown note above)")
